# Notebook 00 — frozen 100K/15K/5K resources

Run once. This notebook does no PAD training.

In [1]:
%pip install -q "insightface==2.0" "onnxruntime"

import insightface
import onnxruntime as ort

print("InsightFace:", insightface.__version__)
print("ONNX Runtime providers:", ort.get_available_providers())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 65.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
InsightFace: 2.0
ONNX Runtime providers: ['AzureExecutionProvider', 'CPUExecutionProvider']


In [2]:
# CONFIG / DATA ROOTS: edit explicit mounts only. No recursive /kaggle/input search.
CELEBA_ROOT=("/kaggle/input/datasets/attentionlayer241/celeba-spoof-for-face-antispoofing/"
             "CelebA_Spoof_/CelebA_Spoof")
LCC_FASD_ROOT="/kaggle/input/datasets/aleksandrpikul222/lcc-fasd/LCC_FASD"
STAGE0_RESOURCE_DIR="/kaggle/input/datasets/maithanhphuong/face-auth-stage0-micro-v1"
SCRFD_MODEL_PATH="/kaggle/input/datasets/maithanhphuong/scrfd-model/det_500m.onnx"
OUTPUT_ROOT="/kaggle/working/minifasnet_final_resources_v1"
DATA_SEED=42; TEST_SEED=4243
TRAIN_N=100000; VAL_N=15000; TEST_N=5000; SCALE_TRAIN_N=20000; SCALE_VAL_N=5000
CACHE_SAVE_EVERY=5000
CROP_SCALE=2.7; INPUT_SIZE=80


In [3]:
# Dependency and exact detector preflight; run before data sampling or SCRFD work.
import sys, importlib.util, hashlib
from pathlib import Path
required=('cv2','numpy','pandas','sklearn','insightface','onnxruntime')
missing=[name for name in required if importlib.util.find_spec(name) is None]
if missing:raise RuntimeError('Install required Kaggle packages before Notebook 00: '+', '.join(missing))
import cv2, numpy as np, pandas as pd, sklearn, insightface, onnxruntime as ort
from insightface import model_zoo
EXPECTED_SCRFD_SHA256='5e4447f50245bbd7966bd6c0fa52938c61474a04ec7def48753668a9d8b4ea3a'
EXPECTED_INSIGHTFACE_VERSION='2.0' # frozen E1 v5.3 / Stage0 detector environment
if str(getattr(insightface,'__version__','unknown'))!=EXPECTED_INSIGHTFACE_VERSION:
    raise RuntimeError(f'InsightFace {EXPECTED_INSIGHTFACE_VERSION} required; found {getattr(insightface,"__version__","unknown")}')
providers=ort.get_available_providers()
if 'CPUExecutionProvider' not in providers:raise RuntimeError('ONNX Runtime CPUExecutionProvider is required')
model_path=Path(SCRFD_MODEL_PATH)
if not model_path.is_file():raise FileNotFoundError('Exact SCRFD-500M model missing: '+str(model_path))
def preflight_sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda:stream.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()
model_sha=preflight_sha256(model_path)
if model_sha!=EXPECTED_SCRFD_SHA256:raise RuntimeError(f'SCRFD SHA mismatch: {model_sha}')
print('Python:',sys.version.split()[0],'OpenCV:',cv2.__version__,'NumPy:',np.__version__)
print('Pandas:',pd.__version__,'scikit-learn:',sklearn.__version__)
print('InsightFace:',insightface.__version__,'ONNX Runtime:',ort.__version__,'providers:',providers)
print('SCRFD-500M SHA256:',model_sha,'threshold 0.50; sizes 640, 480, 320')


Python: 3.12.13 OpenCV: 4.13.0 NumPy: 2.0.2
Pandas: 2.3.3 scikit-learn: 1.6.1
InsightFace: 2.0 ONNX Runtime: 1.30.0 providers: ['AzureExecutionProvider', 'CPUExecutionProvider']
SCRFD-500M SHA256: 5e4447f50245bbd7966bd6c0fa52938c61474a04ec7def48753668a9d8b4ea3a threshold 0.50; sizes 640, 480, 320


In [4]:
import csv, hashlib, importlib.util, json, math, os, random, shutil, time, zipfile, copy, gc
from collections import Counter, OrderedDict
from pathlib import Path, PurePosixPath
import cv2, numpy as np, pandas as pd, torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import confusion_matrix, roc_curve
from IPython.display import FileLink, display
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def sha256_text(value):
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def ordered_path_fingerprint(paths):
    return sha256_text("\n".join(map(str, paths)))


def assert_ordered_keys(actual, expected, name):
    actual = list(map(str, actual))
    expected = list(map(str, expected))
    if actual != expected:
        raise RuntimeError(
            "{} sample keys/order differ from the frozen manifest: actual_n={}, expected_n={}".format(
                name, len(actual), len(expected)
            )
        )


def bbox_jitter(record):
    x1, y1, x2, y2 = map(float, record["bbox_xyxy"])
    w, h = x2 - x1, y2 - y1
    if random.random() < 0.20:
        base = max(w, h)
        cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
        scale = random.uniform(0.95, 1.05)
        cx += random.uniform(-0.05, 0.05) * base
        cy += random.uniform(-0.05, 0.05) * base
        w, h = w * scale, h * scale
        return [cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2]
    return [x1, y1, x2, y2]


def apply_e1_mild_augmentation(rgb):
    if random.random() < 0.5:
        rgb = cv2.flip(rgb, 1)
    if random.random() < 0.30:
        alpha = random.uniform(0.90, 1.10)
        beta = random.uniform(-0.10, 0.10) * 255.0
        rgb = np.clip(rgb.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)
    return rgb


def binary_score(logits):
    logits = torch.as_tensor(logits)
    if logits.ndim != 2 or logits.shape[1] != 3:
        raise ValueError("PAD logits must have shape [N, 3] for Real/Physical/Digital")
    if not torch.isfinite(logits).all():
        raise ValueError("PAD logits contain NaN or Infinity")
    return logits[:, 0] - torch.logsumexp(logits[:, 1:], dim=1)


def safe_auc(y_real, scores):
    from sklearn.metrics import roc_auc_score
    y_real = np.asarray(y_real, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    if y_real.ndim != 1 or scores.ndim != 1 or len(y_real) != len(scores) or len(y_real) == 0:
        raise ValueError("AUC needs non-empty one-dimensional labels and scores of equal length")
    if not np.isfinite(scores).all():
        raise ValueError("AUC scores contain NaN or Infinity")
    if set(map(int, np.unique(y_real))) != {0, 1}:
        raise ValueError("AUC requires both binary classes (positive class is Real)")
    return float(roc_auc_score(y_real, scores))


def metrics_binary(y3, scores, threshold, coverage=None, candidate_n=None):
    y3 = np.asarray(y3, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    if y3.ndim != 1 or scores.ndim != 1 or len(y3) != len(scores) or len(y3) == 0:
        raise ValueError("PAD metrics need non-empty 1-D labels/scores with equal lengths")
    if not set(map(int, np.unique(y3))).issubset({0, 1, 2}):
        raise ValueError("PAD labels must be from the frozen class mapping 0/1/2")
    if not np.isfinite(scores).all() or not np.isfinite(float(threshold)):
        raise ValueError("PAD scores and threshold must be finite")
    y_real = (y3 == 0).astype(np.int64)
    if set(map(int, np.unique(y_real))) != {0, 1}:
        raise ValueError("PAD metrics require at least one Real and one Attack sample")
    if coverage is not None and (not np.isfinite(float(coverage)) or not 0.0 <= float(coverage) <= 1.0):
        raise ValueError("Detector coverage must be finite and within [0, 1]")
    scored_n = len(y3)
    candidate_n = int(candidate_n if candidate_n is not None else scored_n)
    if candidate_n < scored_n:
        raise ValueError("candidate_n cannot be less than n_scored")
    pred_real = scores >= float(threshold)
    real = y_real == 1
    attack = y_real == 0
    bpcer = float(np.mean(~pred_real[real]))
    apcer = float(np.mean(pred_real[attack]))
    acer = 0.5 * (apcer + bpcer)
    return {
        "n_scored": int(scored_n),
        "candidate_n": candidate_n,
        "detector_coverage": float(coverage if coverage is not None else 1.0),
        "apcer": apcer,
        "bpcer": bpcer,
        "acer": acer,
        "hter": acer,
        "auc": safe_auc(y_real, scores),
        "accuracy": float(np.mean(pred_real == y_real)),
        "locked_logit_threshold": float(threshold),
    }


def metrics_celeba(y3, logits, threshold):
    scores = binary_score(torch.as_tensor(logits)).cpu().numpy()
    result = metrics_binary(y3, scores, threshold, coverage=1.0, candidate_n=len(y3))
    result["accuracy_3class"] = float(np.mean(np.argmax(logits, axis=1) == np.asarray(y3)))
    return result, scores


def make_prediction_frame(keys, y3, codes, logits, threshold):
    scores = binary_score(torch.as_tensor(logits)).cpu().numpy()
    pred_real = scores >= threshold
    result = pd.DataFrame({
        "key": list(map(str, keys)),
        "true_class": np.asarray(y3, dtype=np.int64),
        "attack_code": np.asarray(codes, dtype=np.int64),
        "logit_real": logits[:, 0],
        "logit_physical": logits[:, 1],
        "logit_digital": logits[:, 2],
        "pad_logit_score": scores,
        "pred_real": pred_real,
        "correct_binary": pred_real == (np.asarray(y3) == 0),
        "pred_class_3": np.argmax(logits, axis=1),
        "correct_3class": np.argmax(logits, axis=1) == np.asarray(y3),
    })
    return result


def eval_spatial(model, loader, device, amp):
    model.eval()
    logits_all, y_all, keys_all, codes_all = [], [], [], []
    with torch.no_grad():
        for x, y, keys, codes in loader:
            x = x.to(device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=amp):
                logits = model(x)
            logits_all.append(logits.float().cpu().numpy())
            y_all.extend(np.asarray(y, dtype=np.int64).tolist())
            keys_all.extend(list(keys))
            codes_all.extend(np.asarray(codes, dtype=np.int64).tolist())
    if not logits_all:
        raise RuntimeError("Spatial evaluation loader produced no samples")
    logits = np.concatenate(logits_all)
    labels = np.asarray(y_all, dtype=np.int64)
    codes = np.asarray(codes_all, dtype=np.int64)
    if len(logits) != len(labels) or len(keys_all) != len(labels) or len(codes) != len(labels):
        raise RuntimeError("Spatial evaluation outputs have inconsistent sample counts")
    return logits, labels, list(map(str, keys_all)), codes


def eval_lcc_spatial(model, loader, device, amp, lcc_frame, threshold):
    model.eval()
    logits_all, y_all, keys_all = [], [], []
    with torch.no_grad():
        for x, y, keys, _status in loader:
            x = x.to(device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=amp):
                logits = model(x)
            logits_all.append(logits.float().cpu().numpy())
            y_all.extend(np.asarray(y, dtype=np.int64).tolist())
            keys_all.extend(list(keys))
    if not logits_all:
        raise RuntimeError("Common LCC VALID evaluation loader produced no samples")
    logits = np.concatenate(logits_all)
    y = np.asarray(y_all, dtype=np.int64)
    keys = list(map(str, keys_all))
    expected_paths = lcc_frame.loc[lcc_frame["status"].astype(str) == "VALID", "path"].astype(str).tolist()
    assert_ordered_keys(keys, expected_paths, "Stage 1 LCC")
    if len(logits) != len(y) or len(keys) != len(y):
        raise RuntimeError("Stage 1 LCC logits/labels/keys have inconsistent counts")
    scores = binary_score(torch.as_tensor(logits)).cpu().numpy()
    scored = pd.DataFrame({
        "path": keys, "label": y,
        "logit_real": logits[:, 0],
        "logit_physical": logits[:, 1],
        "logit_digital": logits[:, 2],
        "pad_logit_score": scores,
    })
    full = lcc_frame[["path", "label", "status"]].merge(
        scored, on=["path", "label"], how="left", validate="one_to_one"
    )
    full["dataset_role"] = "LCC-FASD external-dev / cross-domain stress set"
    valid = full["status"].astype(str).eq("VALID")
    if full.loc[valid, "pad_logit_score"].isna().any() or int(valid.sum()) != len(keys):
        raise RuntimeError("Stage 1 inference failed to score the exact common LCC VALID set")
    full["pred_real"] = np.nan
    full.loc[valid, "pred_real"] = full.loc[valid, "pad_logit_score"] >= float(threshold)
    full["pred_real"] = full["pred_real"].where(valid, np.nan)
    return logits, y, keys, full


def write_zip(folder, zip_path):
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(Path(folder).rglob("*")):
            if path.is_file() and path.resolve() != Path(zip_path).resolve():
                archive.write(path, arcname=Path(Path(folder).name) / path.relative_to(folder))


def strict_json_write(path, payload):
    Path(path).write_text(json.dumps(payload, indent=2, allow_nan=False)+'\n')

def ordered_sha(keys):
    return sha256_text('\n'.join(map(str,keys)))

def atomic_json_write(path,payload):
    path=Path(path);tmp=path.with_suffix(path.suffix+'.tmp')
    strict_json_write(tmp,payload);os.replace(tmp,path)

def threshold_from_val(y3,scores):
    # Exact minimum ACER over lower/midpoint/upper candidates, computed in O(N log N).
    y=np.asarray(y3,dtype=np.int64);s=np.asarray(scores,dtype=np.float64)
    if len(y)!=len(s) or not len(y) or not np.isfinite(s).all() or set(np.unique(y))!={0,1,2}:
        raise ValueError('Val threshold requires finite scores and all three classes')
    order=np.argsort(s,kind='mergesort');v=s[order];real=(y[order]==0)
    n_real=int(real.sum());n_attack=len(y)-n_real
    boundaries=np.r_[0,np.flatnonzero(np.diff(v))+1,len(v)]
    real_prefix=np.r_[0,np.cumsum(real)];attack_prefix=np.r_[0,np.cumsum(~real)]
    candidates=[]
    for j in range(len(boundaries)):
        k=int(boundaries[j]);bpcer=real_prefix[k]/n_real;apcer=(n_attack-attack_prefix[k])/n_attack
        if j==0:thr=float(np.nextafter(v[0],-np.inf))
        elif j==len(boundaries)-1:thr=float(np.nextafter(v[-1],np.inf))
        else:thr=float(v[k-1]/2+v[k]/2)
        candidates.append((.5*(apcer+bpcer),abs(thr),thr))
    return min(candidates)[2]
SCRFD_PRIMARY_INPUT_SIZE=(640,640)
SCRFD_FALLBACK_INPUT_SIZES=[(480,480),(320,320)]
SCRFD_CONF_THRESH=.5
RELIABLE_RAW_IOU_MIN=.5
RELIABLE_CENTER_SHIFT_MAX=.2
def label_of(meta_value):
    """Map CelebA-Spoof detailed attack type at index 40 -> 3-class label."""
    if len(meta_value) <= 40:
        raise ValueError(f"Metadata entry too short: len={len(meta_value)}")
    raw = int(meta_value[40])
    if raw == 0:
        return 0
    if raw in [1, 2, 3, 4, 5, 6, 7]:
        return 1
    return 2


def subject_id_from_relpath(rel_path):
    parts = PurePosixPath(rel_path.replace("\\", "/")).parts
    for i, part in enumerate(parts[:-1]):
        if part.lower() in {"train", "test"} and i + 1 < len(parts):
            return parts[i + 1]
    raise ValueError(f"Cannot infer subject id from path: {rel_path}")


def bbox_wh(b):
    x1, y1, x2, y2 = map(float, b)
    return np.array([x2 - x1, y2 - y1], dtype=np.float64)


def bbox_area(b):
    w, h = bbox_wh(b)
    return max(0.0, float(w)) * max(0.0, float(h))


def bbox_center(b):
    x1, y1, x2, y2 = map(float, b)
    return np.array([(x1 + x2) / 2.0, (y1 + y2) / 2.0], dtype=np.float64)


def bbox_intersection_area(a, b):
    ax1, ay1, ax2, ay2 = map(float, a)
    bx1, by1, bx2, by2 = map(float, b)
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    return max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)


def bbox_iou(a, b):
    inter = bbox_intersection_area(a, b)
    union = bbox_area(a) + bbox_area(b) - inter
    return 0.0 if union <= 0 else float(inter / union)


def normalized_center_shift(reference, candidate):
    ref_wh = bbox_wh(reference)
    denom = max(float(ref_wh[0]), float(ref_wh[1]), 1e-8)
    return float(np.linalg.norm(bbox_center(reference) - bbox_center(candidate)) / denom)


def read_celeba_bbox_xyxy(img_path, image_shape=None):
    """Convert CelebA-Spoof 224-reference *_BB.txt coordinates to native-image xyxy."""
    if image_shape is None:
        image = cv2.imread(img_path, cv2.IMREAD_COLOR)
        if image is None:
            raise FileNotFoundError(f"Cannot read image: {img_path}")
        h_img, w_img = image.shape[:2]
    else:
        h_img, w_img = image_shape[:2]

    bb_path = os.path.splitext(img_path)[0] + "_BB.txt"
    if not os.path.exists(bb_path):
        raise FileNotFoundError(f"Missing bbox file: {bb_path}")

    with open(bb_path, "r") as f:
        parts = f.readline().strip().split()
    if len(parts) < 4:
        raise ValueError(f"bbox needs >=4 values, got {parts}")

    x, y, w, h = map(float, parts[:4])
    x = x / 224.0 * w_img
    y = y / 224.0 * h_img
    w = w / 224.0 * w_img
    h = h / 224.0 * h_img
    if w <= 0 or h <= 0:
        raise ValueError(f"Non-positive CelebA bbox at {bb_path}: {(x, y, w, h)}")
    return np.array([x, y, x + w, y + h], dtype=np.float64)
def mini_crop_bgr(image,bbox_xyxy,scale=CROP_SCALE,size=INPUT_SIZE):
    # Official CropImage._get_new_box geometry, adapted from xyxy face coordinates.
    src_h,src_w=image.shape[:2]
    x1,y1,x2,y2=map(float,bbox_xyxy)
    box_w,box_h=x2-x1,y2-y1
    if box_w<=0 or box_h<=0 or src_w<2 or src_h<2:
        raise ValueError('Invalid MiniFASNet image/bbox geometry')
    used_scale=min((src_h-1)/box_h,(src_w-1)/box_w,float(scale))
    new_w,new_h=box_w*used_scale,box_h*used_scale
    center_x,center_y=box_w/2+x1,box_h/2+y1
    left,top=center_x-new_w/2,center_y-new_h/2
    right,bottom=center_x+new_w/2,center_y+new_h/2
    if left<0: right-=left;left=0
    if top<0: bottom-=top;top=0
    if right>src_w-1: left-=right-src_w+1;right=src_w-1
    if bottom>src_h-1: top-=bottom-src_h+1;bottom=src_h-1
    lx,ty,rx,by=map(int,(left,top,right,bottom))
    if lx<0 or ty<0 or rx>=src_w or by>=src_h or rx<lx or by<ty:
        raise RuntimeError(f'MiniFASNet crop escaped image: {(lx,ty,rx,by)}')
    patch=image[ty:by+1,lx:rx+1]
    if patch.size==0:raise RuntimeError('Empty MiniFASNet crop')
    return cv2.resize(patch,(size,size)),(lx,ty,rx,by),used_scale


def mini_tensor(patch):
    if patch.shape!=(INPUT_SIZE,INPUT_SIZE,3) or patch.dtype!=np.uint8:
        raise ValueError('Expected 80x80 uint8 BGR patch')
    return torch.from_numpy(np.ascontiguousarray(patch.transpose(2,0,1))).float().div_(255.)




In [5]:
DATA_ROOT=str(Path(CELEBA_ROOT).expanduser())
root=Path(OUTPUT_ROOT);manifest_dir=root/'manifests';bbox_dir=root/'bbox';figure_dir=root/'figures'
for p in (manifest_dir,bbox_dir,figure_dir):p.mkdir(parents=True,exist_ok=True)
celeba_root=Path(DATA_ROOT);lcc_root=Path(LCC_FASD_ROOT)
train_json=celeba_root/'metas/intra_test/train_label.json';test_json=celeba_root/'metas/intra_test/test_label.json'
for p in (train_json,test_json,lcc_root,Path(SCRFD_MODEL_PATH)):
    if not p.exists():raise FileNotFoundError(p)
train_meta=json.loads(train_json.read_text());test_meta=json.loads(test_json.read_text())
if set(train_meta)&set(test_meta):raise RuntimeError('Official Train/Test key overlap')
source_hashes={'train_label_json':sha256_file(train_json),'test_label_json':sha256_file(test_json),
               'scrfd_model':sha256_file(SCRFD_MODEL_PATH)}
if (root/'provenance.json').is_file():
    previous=json.loads((root/'provenance.json').read_text())
    if previous['source_hashes']!=source_hashes:raise RuntimeError('Frozen source SHA changed')
print('Official populations:',len(train_meta),len(test_meta))
print('Official class counts:',dict(Counter(label_of(v) for v in train_meta.values())),dict(Counter(label_of(v) for v in test_meta.values())))
print('Metadata SHA:',source_hashes)

def stratified(keys,meta,n,seed):
    keys=np.asarray(list(map(str,keys)),dtype=object)
    if len(keys)<n:raise RuntimeError(f'Only {len(keys)} candidates for {n}')
    if len(keys)==n:return list(keys)
    labels=np.asarray([label_of(meta[k]) for k in keys])
    selected,_=train_test_split(keys,train_size=n,stratify=labels,random_state=seed)
    return list(map(str,selected))

all_train=np.asarray(list(train_meta),dtype=object);all_test=np.asarray(list(test_meta),dtype=object)
test_subjects={subject_id_from_relpath(k) for k in all_test}
clean_train=np.asarray([k for k in all_train if subject_id_from_relpath(k) not in test_subjects],dtype=object)
subjects=np.asarray([subject_id_from_relpath(k) for k in clean_train]);labels=np.asarray([label_of(train_meta[k]) for k in clean_train]);
# E1 v5.3: remove official-Test subject overlap, then GroupShuffleSplit by subject.
# Val pool is deliberately larger than 15K so exact, stratified 15K can be frozen.
gss=GroupShuffleSplit(n_splits=64,test_size=max(1.25*VAL_N/len(clean_train),VAL_N/len(clean_train)+.01),random_state=DATA_SEED)
props=np.bincount(labels,minlength=3)/len(labels);choices=[]
for pool_idx,val_idx in gss.split(clean_train,labels,groups=subjects):
    vy=labels[val_idx];vcounts=np.bincount(vy,minlength=3)
    if len(pool_idx)<TRAIN_N or len(val_idx)<VAL_N or np.any(vcounts==0):continue
    # Ensure exact stratified Val15K can be drawn.
    try:stratified(clean_train[val_idx],train_meta,VAL_N,DATA_SEED)
    except ValueError:continue
    objective=abs(len(val_idx)-int(1.25*VAL_N))/VAL_N+5*float(np.abs(vcounts/len(vy)-props).sum())
    choices.append((objective,pool_idx,val_idx))
if not choices:raise RuntimeError('Cannot make subject-disjoint Train100K/Val15K pools')
_,pool_idx,val_idx=min(choices,key=lambda z:z[0])
train_pool=list(map(str,clean_train[pool_idx]))
final_val=stratified(clean_train[val_idx],train_meta,VAL_N,DATA_SEED+1)
final_test=stratified(all_test,test_meta,TEST_N,TEST_SEED)
assert len(final_val)==VAL_N and len(final_test)==TEST_N
assert set(train_pool).isdisjoint(final_val) and set(train_pool).isdisjoint(final_test)
assert {subject_id_from_relpath(k) for k in train_pool}.isdisjoint({subject_id_from_relpath(k) for k in final_val})
assert {subject_id_from_relpath(k) for k in train_pool+final_val}.isdisjoint(test_subjects)
# An old resource build froze Train before E1 train-only filtering. Do not mutate it in place.
if (manifest_dir/'final_train100k.csv').exists() and not (bbox_dir/'train_filter_audit.json').exists():
    raise RuntimeError('Old unfiltered resource output found; use a clean OUTPUT_ROOT to rebuild Notebook 00')
print('Subject-disjoint candidate pools:',len(train_pool),len(clean_train[val_idx]))


Official populations: 494405 67170
Official class counts: {0: 162462, 1: 226715, 2: 105228} {1: 35495, 2: 11752, 0: 19923}
Metadata SHA: {'train_label_json': 'fe96b85f575b98d7c231cbf24eb249014b57e95a9f7c1ebd3624034c30d7a30a', 'test_label_json': '5024238665680fd6df3a1b4b1cf00b75cc60386e6c6978822df86cfe794bd14d', 'scrfd_model': '5e4447f50245bbd7966bd6c0fa52938c61474a04ec7def48753668a9d8b4ea3a'}
Subject-disjoint candidate pools: 474335 19854


In [6]:
def _select_target_detection(bboxes, gt_bbox):
    if bboxes is None or len(bboxes) == 0:
        return None

    candidates = []
    for row in bboxes:
        det = np.asarray(row[:4], dtype=np.float64)
        conf = float(row[4])
        candidates.append({
            "bbox": det,
            "confidence": conf,
            "raw_iou": bbox_iou(gt_bbox, det),
            "center_shift": normalized_center_shift(gt_bbox, det),
            "n_faces": int(len(bboxes)),
        })

    return max(candidates, key=lambda x: (x["raw_iou"], x["confidence"]))


def _scrfd_detect(detector, image_bgr, input_size):
    """
    Prefer detect(..., input_size=...). Older InsightFace variants are handled
    by temporarily changing detector.input_size.
    """
    try:
        return detector.detect(
            image_bgr,
            input_size=tuple(input_size),
            max_num=0,
            metric="default",
        )
    except TypeError:
        old_input_size = getattr(detector, "input_size", None)
        detector.input_size = tuple(input_size)
        try:
            return detector.detect(image_bgr, max_num=0, metric="default")
        finally:
            detector.input_size = old_input_size


def build_cache_record(detector, rel_path):
    img_path = os.path.join(DATA_ROOT, rel_path)
    image = cv2.imread(img_path, cv2.IMREAD_COLOR)
    if image is None:
        return {"status": "INVALID", "reason": "image_read_failure"}

    try:
        gt = read_celeba_bbox_xyxy(img_path, image.shape)
    except Exception as exc:
        return {
            "status": "INVALID",
            "reason": f"celeba_bbox_error:{type(exc).__name__}",
        }

    gt_w, gt_h = bbox_wh(gt)
    gt_min_side = float(min(gt_w, gt_h))

    sizes_to_try = [SCRFD_PRIMARY_INPUT_SIZE] + list(SCRFD_FALLBACK_INPUT_SIZES)
    best_unreliable = None

    for input_size in sizes_to_try:
        bboxes, _ = _scrfd_detect(detector, image, input_size=input_size)
        candidate = _select_target_detection(bboxes, gt)
        if candidate is None:
            continue

        candidate["input_size"] = list(input_size)

        if (
            candidate["raw_iou"] >= RELIABLE_RAW_IOU_MIN
            and candidate["center_shift"] <= RELIABLE_CENTER_SHIFT_MAX
        ):
            det = candidate["bbox"]
            dw, dh = bbox_wh(det)
            return {
                "status": "VALID",
                "bbox_source": f"SCRFD_{input_size[0]}",
                "bbox_xyxy": [float(x) for x in det],
                
                "face_min_side": float(min(dw, dh)),
                "confidence": candidate["confidence"],
                "n_faces": candidate["n_faces"],
                "raw_iou_to_celeba": candidate["raw_iou"],
                "center_shift_norm": candidate["center_shift"],
                "det_input_size": list(input_size),
                "gt_min_side": gt_min_side,
            }

        if best_unreliable is None or (
            candidate["raw_iou"], candidate["confidence"]
        ) > (
            best_unreliable["raw_iou"], best_unreliable["confidence"]
        ):
            best_unreliable = candidate

    # Do not auto-drop detector failures: the diagnostic showed class-dependent
    # failure, especially for Physical Spoof. Keep sample through CelebA fallback.
    if best_unreliable is None:
        return {
            "status": "FALLBACK_DET_FAIL",
            "bbox_source": "CELEBA_FALLBACK_DET_FAIL",
            "bbox_xyxy": [float(x) for x in gt],
            
            "face_min_side": gt_min_side,
            "confidence": None,
            "n_faces": 0,
            "raw_iou_to_celeba": None,
            "center_shift_norm": None,
            "det_input_size": None,
            "gt_min_side": gt_min_side,
        }

    # Suspicious geometry: retain Val/Test by GT fallback, but Train filters it.
    return {
        "status": "FALLBACK_SUSPICIOUS",
        "bbox_source": "CELEBA_FALLBACK_SUSPICIOUS",
        "bbox_xyxy": [float(x) for x in gt],
        
        "face_min_side": gt_min_side,
        "confidence": best_unreliable["confidence"],
        "n_faces": best_unreliable["n_faces"],
        "raw_iou_to_celeba": best_unreliable["raw_iou"],
        "center_shift_norm": best_unreliable["center_shift"],
        "det_input_size": best_unreliable["input_size"],
        "gt_min_side": gt_min_side,
    }
from tqdm.auto import tqdm
TRAIN_MIN_FACE_PX=48
MAX_TOTAL_TRAIN_FILTER_RATE=.05
MAX_CLASS_TRAIN_FILTER_RATE=.08
MAX_CLASS_FILTER_GAP=.03
MAX_SUSPICIOUS_TRAIN_RATE=.01
if source_hashes['scrfd_model']!=EXPECTED_SCRFD_SHA256:raise RuntimeError('Detector changed after preflight')
provider=['CUDAExecutionProvider','CPUExecutionProvider'] if 'CUDAExecutionProvider' in providers else ['CPUExecutionProvider']
ctx=0 if provider[0]=='CUDAExecutionProvider' else -1
policy={'source':'E1_v5_3','model_sha256':source_hashes['scrfd_model'],
    'insightface_version':insightface.__version__,'sizes':[[640,640],[480,480],[320,320]],
    'det_thresh':SCRFD_CONF_THRESH,'raw_iou_min':RELIABLE_RAW_IOU_MIN,
    'center_shift_max':RELIABLE_CENTER_SHIFT_MAX,'fallback':'official CelebA annotation',
    'records':'face localization only; E1 crop factors removed','provider':provider}
cache_path=bbox_dir/'celeba_final_bbox_cache.json';progress_path=bbox_dir/'celeba_bbox_progress.json'
audit_path=bbox_dir/'train_filter_audit.json'
valid_statuses={'VALID','FALLBACK_DET_FAIL','FALLBACK_SUSPICIOUS'}

def train_filter_reason(record):
    status=record.get('status')
    if status=='FALLBACK_SUSPICIOUS':return 'SUSPICIOUS_GEOMETRY'
    if status not in ('VALID','FALLBACK_DET_FAIL'):return 'INVALID_LOCALIZATION'
    box=record.get('bbox_xyxy')
    if box is None or len(box)!=4:return 'INVALID_LOCALIZATION'
    x1,y1,x2,y2=map(float,box)
    if not all(np.isfinite((x1,y1,x2,y2))) or x2<=x1 or y2<=y1:return 'INVALID_LOCALIZATION'
    if min(x2-x1,y2-y1)<TRAIN_MIN_FACE_PX:return 'TOO_SMALL'
    return None

def make_train_audit(candidates,records,final_keys=()):
    statuses=Counter(records[k].get('status','UNKNOWN') for k in candidates)
    reasons={k:train_filter_reason(records[k]) for k in candidates}
    filtered=[k for k in candidates if reasons[k] is not None]
    eligible=[k for k in candidates if reasons[k] is None]
    candidate_counts=Counter(label_of(train_meta[k]) for k in candidates)
    filtered_counts=Counter(label_of(train_meta[k]) for k in filtered)
    final_counts=Counter(label_of(train_meta[k]) for k in final_keys)
    rates={str(c):filtered_counts[c]/candidate_counts[c] if candidate_counts[c] else 0. for c in range(3)}
    total_rate=len(filtered)/len(candidates) if candidates else 0.
    small=int(sum(bool(min(bbox_wh(records[k]['bbox_xyxy']))<TRAIN_MIN_FACE_PX)
        for k in candidates if records[k].get('bbox_xyxy') is not None))
    suspicious=statuses['FALLBACK_SUSPICIOUS'];gap=max(rates.values())-min(rates.values())
    audit={'candidate_train_evaluated':len(candidates),'status_counts':dict(statuses),
        'valid_count':statuses['VALID'],'fallback_det_fail_count':statuses['FALLBACK_DET_FAIL'],
        'fallback_suspicious_count':suspicious,'small_face_lt_48_count':small,
        'suspicious_geometry_count':suspicious,'filter_reason_counts':dict(Counter(reasons[k] for k in filtered)),
        'eligible_count':len(eligible),'final_sampled_train_count':len(final_keys),
        'per_class_candidate_counts':{str(c):candidate_counts[c] for c in range(3)},
        'per_class_filtered_counts':{str(c):filtered_counts[c] for c in range(3)},
        'per_class_final_counts':{str(c):final_counts[c] for c in range(3)},
        'overall_filter_rate':total_rate,'per_class_filter_rate':rates,'class_filter_gap':gap,
        'suspicious_geometry_rate':suspicious/len(candidates) if candidates else 0.,
        'guardrails':{'overall_max':MAX_TOTAL_TRAIN_FILTER_RATE,'per_class_max':MAX_CLASS_TRAIN_FILTER_RATE,
            'class_gap_max':MAX_CLASS_FILTER_GAP,'suspicious_max':MAX_SUSPICIOUS_TRAIN_RATE},
        'rule':'TRAIN only: drop suspicious geometry, invalid localization, or bbox minimum side <48 native pixels'}
    strict_json_write(audit_path,audit)
    print('Train filter rates:',{'overall':total_rate,'Real':rates['0'],'Physical':rates['1'],
        'Digital':rates['2'],'class_gap':gap,'suspicious':audit['suspicious_geometry_rate']})
    if (total_rate>MAX_TOTAL_TRAIN_FILTER_RATE or max(rates.values())>MAX_CLASS_TRAIN_FILTER_RATE
        or gap>MAX_CLASS_FILTER_GAP or audit['suspicious_geometry_rate']>MAX_SUSPICIOUS_TRAIN_RATE):
        raise RuntimeError('E1 Train-filter bias guardrail exceeded; inspect '+str(audit_path))
    return eligible,audit

if cache_path.exists():
    payload=json.loads(cache_path.read_text())
    if payload.get('policy')!=policy or not audit_path.exists():
        raise RuntimeError('Existing final cache has incompatible or pre-hygiene provenance; use clean OUTPUT_ROOT')
    records=payload['records']
    final_train=pd.read_csv(manifest_dir/'final_train100k.csv').sample_key.astype(str).tolist()
    train_filter_audit=json.loads(audit_path.read_text())
    if (train_filter_audit.get('final_sampled_train_count')!=TRAIN_N or
        sum(train_filter_audit.get('per_class_final_counts',{}).values())!=TRAIN_N):
        raise RuntimeError('Frozen Train filter audit is incomplete')
    if len(final_train)!=TRAIN_N or any(train_filter_reason(records[k]) is not None for k in final_train):
        raise RuntimeError('Frozen final Train does not satisfy TRAIN-only hygiene')
else:
    progress=json.loads(progress_path.read_text()) if progress_path.exists() else {'policy':policy,'records':{}}
    if progress.get('policy')!=policy or not set(progress.get('records',{}))<=set(train_pool)|set(final_val)|set(final_test):
        raise RuntimeError('Bbox progress policy/keys differ')
    records=progress['records'];detector=None
    def ensure_bbox(keys):
        global detector
        missing=[k for k in keys if k not in records]
        if missing and detector is None:
            detector=model_zoo.get_model(str(Path(SCRFD_MODEL_PATH)),providers=provider)
            if detector is None:raise RuntimeError('Exact SCRFD-500M model cannot be loaded')
            detector.prepare(ctx_id=ctx,input_size=SCRFD_PRIMARY_INPUT_SIZE,det_thresh=SCRFD_CONF_THRESH)
        for i,key in enumerate(tqdm(missing,desc='E1 SCRFD bbox'),1):
            records[key]=build_cache_record(detector,key)
            if i%CACHE_SAVE_EVERY==0:atomic_json_write(progress_path,{'policy':policy,'records':records})
        if missing:atomic_json_write(progress_path,{'policy':policy,'records':records})
    # Start with stratified headroom, then extend deterministically only if needed.
    candidate_n=min(len(train_pool),TRAIN_N+10000)
    train_candidates=stratified(train_pool,train_meta,candidate_n,DATA_SEED)
    used=set(train_candidates);remaining=[k for k in train_pool if k not in used]
    ensure_bbox(train_candidates)
    eligible,train_filter_audit=make_train_audit(train_candidates,records)
    expansion=0
    while len(eligible)<TRAIN_N:
        if not remaining:raise RuntimeError(f'Only {len(eligible)} eligible Train images; same-subject-pool expansion exhausted')
        expansion+=1;batch=stratified(remaining,train_meta,min(10000,len(remaining)),DATA_SEED+100+expansion)
        used.update(batch);remaining=[k for k in remaining if k not in used]
        train_candidates.extend(batch);ensure_bbox(batch)
        eligible,train_filter_audit=make_train_audit(train_candidates,records)
    final_train=stratified(eligible,train_meta,TRAIN_N,DATA_SEED)
    eligible_set=set(eligible)
    assert set(final_train)<=eligible_set and len(final_train)==TRAIN_N
    _,train_filter_audit=make_train_audit(train_candidates,records,final_train)
    ensure_bbox(final_val+final_test)
    if detector is not None:del detector;gc.collect()

scale_train=stratified(final_train,train_meta,SCALE_TRAIN_N,DATA_SEED+2)
scale_val=stratified(final_val,train_meta,SCALE_VAL_N,DATA_SEED+3)
assert (len(final_train),len(final_val),len(final_test),len(scale_train),len(scale_val))==(100000,15000,5000,20000,5000)
assert set(scale_train)<=set(final_train) and set(scale_val)<=set(final_val)
assert set(final_train).isdisjoint(final_val) and set(final_train).isdisjoint(final_test) and set(final_val).isdisjoint(final_test)
assert {subject_id_from_relpath(k) for k in final_train}.isdisjoint({subject_id_from_relpath(k) for k in final_val})
assert {subject_id_from_relpath(k) for k in final_train+final_val}.isdisjoint(test_subjects)
assert all(train_filter_reason(records[k]) is None for k in final_train)
selected=final_train+final_val+final_test
assert len(selected)==120000 and len(set(selected))==120000 and set(selected)<=set(records)
status_counts=Counter(records[k].get('status') for k in selected)
strict_json_write(bbox_dir/'celeba_bbox_status_audit.json',{'status_counts':dict(status_counts),
    'invalid_examples':[(k,records[k]) for k in selected if records[k].get('status') not in valid_statuses][:20]})
if any(records[k].get('status') not in valid_statuses for k in selected):
    raise RuntimeError('Selected Val/Test image unusable after E1 fallback; inspect bbox status audit')
final_records={k:records[k] for k in selected}
if any('crop_factor' in r for r in final_records.values()):raise RuntimeError('Localization cache must not bake crop factor')
if cache_path.exists():
    if set(records)!=set(selected):raise RuntimeError('Frozen final cache keys differ')
else:atomic_json_write(cache_path,{'schema':'celeba_final_localization_v1','policy':policy,'records':final_records})
records=final_records
if progress_path.exists():progress_path.unlink()
statuses=status_counts
print('CelebA bbox statuses:',statuses,'SHA',sha256_file(cache_path))

manifests={'final_train100k':(final_train,train_meta,'train',DATA_SEED),
           'final_val15k':(final_val,train_meta,'train',DATA_SEED+1),
           'final_test5k':(final_test,test_meta,'test',TEST_SEED),
           'scale_train20k':(scale_train,train_meta,'train',DATA_SEED+2),
           'scale_val5k':(scale_val,train_meta,'train',DATA_SEED+3)}
fingerprints={};summary={}
for name,(keys,meta,split,seed) in manifests.items():
    frame=pd.DataFrame([{'sample_key':k,'three_class_label':label_of(meta[k]),'attack_code':int(meta[k][40]),
        'official_split':split,'sampling_seed':seed,'subject_id':subject_id_from_relpath(k)} for k in keys])
    if len(frame.sample_key.unique())!=len(frame) or set(frame.three_class_label)!={0,1,2}:raise RuntimeError(name+' duplicate key or missing class')
    path=manifest_dir/(name+'.csv')
    csv_text=frame.to_csv(index=False)
    if path.exists() and path.read_text()!=csv_text:
        raise RuntimeError('Previously frozen manifest differs: '+name)
    path.write_text(csv_text)
    source_sha=source_hashes[split+'_label_json']
    fingerprints[name]={'sample_count':len(frame),'class_counts':{str(k):int(v) for k,v in frame.three_class_label.value_counts().items()},
        'ordered_sample_key_sha256':ordered_sha(keys),'sampling_seed':seed,'source_metadata_sha256':source_sha,'file_sha256':sha256_file(path)}
    summary[name]=fingerprints[name]
strict_json_write(manifest_dir/'manifest_fingerprints.json',fingerprints)
strict_json_write(manifest_dir/'manifest_summary.json',summary)
for name,(keys,_,_,_) in manifests.items():
    if len(keys)!=fingerprints[name]['sample_count']:raise RuntimeError('Manifest drift')
print({k:v['sample_count'] for k,v in fingerprints.items()})


E1 SCRFD bbox:   0%|          | 0/110000 [00:00<?, ?it/s]

Train filter rates: {'overall': 0.011127272727272727, 'Real': 0.014687984067271521, 'Physical': 0.0018238407707709692, 'Digital': 0.02567827387310404, 'class_gap': 0.02385443310233307, 'suspicious': 0.0015818181818181818}
Train filter rates: {'overall': 0.011127272727272727, 'Real': 0.014687984067271521, 'Physical': 0.0018238407707709692, 'Digital': 0.02567827387310404, 'class_gap': 0.02385443310233307, 'suspicious': 0.0015818181818181818}


E1 SCRFD bbox:   0%|          | 0/20000 [00:00<?, ?it/s]

CelebA bbox statuses: Counter({'VALID': 118855, 'FALLBACK_DET_FAIL': 1107, 'FALLBACK_SUSPICIOUS': 38}) SHA e15411af09c64581a7567c98e62b97df95bcfde2b3b5affa4af0761f8908ca28
{'final_train100k': 100000, 'final_val15k': 15000, 'final_test5k': 5000, 'scale_train20k': 20000, 'scale_val5k': 5000}


In [7]:
stage0=Path(STAGE0_RESOURCE_DIR)
if not (stage0/'micro_resource_fingerprints.json').is_file() and (stage0/'stage0').is_dir():stage0=stage0/'stage0'
lcc_manifest=manifest_dir/'lcc_external_dev.csv';lcc_cache_path=bbox_dir/'lcc_bbox_cache.json'
if (stage0/'micro_resource_fingerprints.json').is_file():
    fp=json.loads((stage0/'micro_resource_fingerprints.json').read_text())
    source_manifest=stage0/'lcc_external_dev_manifest.csv';source_cache=stage0/'lcc_scrfd_cache_micro_v1.json'
    if sha256_file(source_manifest)!=fp['lcc_manifest_sha256'] or sha256_file(source_cache)!=fp['lcc_cache_sha256']:
        raise RuntimeError('Stage0 LCC SHA verification failed')
    payload=json.loads(source_cache.read_text());source_frame=pd.read_csv(source_manifest,keep_default_na=False)
    if payload['policy']['detector_sha256']!=source_hashes['scrfd_model']:raise RuntimeError('Stage0 detector differs')
    recs=payload['records'];joined={str(r.path):recs[str(r.record_id)] for r in source_frame.itertuples(index=False)}
    if len(joined)!=len(source_frame):raise RuntimeError('LCC path join not unique')
    if any(joined[str(r.path)]['status']!=str(r.status) for r in source_frame.itertuples(index=False)):
        raise RuntimeError('LCC status disagreement')
    # Keep original source file hash and all valid rows. Remove old 1.55 crop metadata.
    source_frame.to_csv(lcc_manifest,index=False)
    lcc_records={path:{k:v for k,v in rec.items() if k!='crop_factor'} for path,rec in joined.items()}
    lcc_policy={k:v for k,v in payload['policy'].items() if k!='crop_factor'}
    lcc_origin={'mode':'verified Stage0 reuse','source_manifest_sha256':fp['lcc_manifest_sha256'],
                'source_cache_sha256':fp['lcc_cache_sha256'],'source_detector_sha256':fp['lcc_detector_sha256']}
else:
    # Stage0 is unavailable. Recreate the same external_scrfd_v1 face selection on explicit LCC lists.
    rows=[]
    for name,label in [('CLIENT_TEST.txt',0),('IMPOSTER_TEST.txt',1)]:
        file=lcc_root/name
        if not file.is_file():raise FileNotFoundError('Missing Stage0 bundle and LCC list: '+str(file))
        for line in file.read_text().splitlines():
            if line.strip():rows.append({'path':line.split()[0].replace('\\','/'),'label':label})
    if len({r['path'] for r in rows})!=len(rows):raise RuntimeError('Duplicate LCC paths')
    lcc_records={p:{'path':p,'label':int(r['label']),'status':'PENDING'} for r in rows for p in [r['path']]}
    detector=model_zoo.get_model(str(Path(SCRFD_MODEL_PATH)),providers=['CPUExecutionProvider'])
    for size in [(640,640),(480,480),(320,320)]:
        detector.prepare(ctx_id=-1,input_size=size,det_thresh=.5)
        for path,rec in tqdm(lcc_records.items(),desc=f'LCC SCRFD {size}'):
            if rec['status']!='PENDING':continue
            image=cv2.imread(str(lcc_root/path),cv2.IMREAD_COLOR)
            if image is None:rec['status']='MISSING_IMAGE';continue
            detections,_=detector.detect(image,max_num=0,metric='default')
            valid=[]
            for det in (detections if detections is not None else []):
                x1,y1,x2,y2,conf=map(float,det[:5])
                if x2>x1 and y2>y1:valid.append(((x2-x1)*(y2-y1),conf,(x1,y1,x2,y2)))
            if valid:
                valid.sort(key=lambda v:(-v[0],-v[1],*v[2]))
                _,confidence,box=valid[0]
                rec.update(status='VALID',bbox_xyxy=list(box),confidence=confidence,n_faces=len(valid),
                           ambiguous_multiface=len(valid)>1,det_input_size=list(size))
    for rec in lcc_records.values():
        if rec['status']=='PENDING':rec['status']='NO_VALID_FACE'
    source_frame=pd.DataFrame([{'path':r['path'],'label':r['label'],'status':lcc_records[r['path']]['status']} for r in rows])
    source_frame.to_csv(lcc_manifest,index=False)
    lcc_policy={'schema':'external_scrfd_v1','detector_sha256':source_hashes['scrfd_model'],
        'sizes':[[640,640],[480,480],[320,320]],'confidence':.5,
        'selection':'largest valid face; confidence then xyxy tie-break'}
    lcc_origin={'mode':'rebuilt once from LCC official TEST lists'}
    del detector
strict_json_write(lcc_cache_path,{'schema':'lcc_final_localization_v1','policy':lcc_policy,'records':lcc_records})
assert set(source_frame.path.astype(str))==set(lcc_records)
assert all((lcc_root/str(p)).is_file() for p in source_frame.path)
valid=source_frame[source_frame.status.astype(str)=='VALID']
assert len(valid)>0 and set(valid.label.astype(int))=={0,1}
lcc_audit={'candidate_count':len(source_frame),'valid_count':len(valid),
    'class_counts':{str(k):int(v) for k,v in valid.label.astype(int).value_counts().items()},
    'ordered_path_sha256':ordered_sha(source_frame.path.astype(str)),
    'bbox_cache_sha256':sha256_file(lcc_cache_path),'detector_sha256':source_hashes['scrfd_model'],**lcc_origin}
strict_json_write(manifest_dir/'lcc_resource_audit.json',lcc_audit)
print('LCC:',lcc_audit)


LCC: {'candidate_count': 3766, 'valid_count': 3762, 'class_counts': {'1': 3373, '0': 389}, 'ordered_path_sha256': '7a4946f4bc8c2a31cbf535f804f946fe873b262810af12d3a0ff9a98bf1187af', 'bbox_cache_sha256': 'd6508bde1c7f947ba7c995340c166bde9dee96b77794152a8bb51cbc56fcfbcf', 'detector_sha256': '5e4447f50245bbd7966bd6c0fa52938c61474a04ec7def48753668a9d8b4ea3a', 'mode': 'verified Stage0 reuse', 'source_manifest_sha256': 'af864ab57bf26af800ceac46c00d9724c8c60f5d0c63037394a1d8708a35abe5', 'source_cache_sha256': '277993f3c8ac436c7fc4f852892a64299b6ccb23063b1774e055a4317c29f78c', 'source_detector_sha256': '5e4447f50245bbd7966bd6c0fa52938c61474a04ec7def48753668a9d8b4ea3a'}


In [8]:
# Galleries verify raw, bbox, actual crop geometry and final BGR tensor.
def gallery_celeba():
    fig,axes=plt.subplots(3,4,figsize=(14,10));rng=np.random.default_rng(100)
    for cls in range(3):
        pool=[k for k in final_train if label_of(train_meta[k])==cls];key=str(rng.choice(pool))
        image=cv2.imread(str(celeba_root/key));rec=records[key]
        patch,box,used=mini_crop_bgr(image,rec['bbox_xyxy'])
        overlay=cv2.cvtColor(image,cv2.COLOR_BGR2RGB);cv2.rectangle(overlay,tuple(map(int,rec['bbox_xyxy'][:2])),tuple(map(int,rec['bbox_xyxy'][2:])),(255,0,0),2)
        views=[cv2.cvtColor(image,cv2.COLOR_BGR2RGB),overlay,cv2.cvtColor(image[box[1]:box[3]+1,box[0]:box[2]+1],cv2.COLOR_BGR2RGB),cv2.cvtColor(patch,cv2.COLOR_BGR2RGB)]
        for j,v in enumerate(views):axes[cls,j].imshow(v);axes[cls,j].axis('off');axes[cls,j].set_title(f'{cls}: '+['raw','bbox',f'2.7 crop eff {used:.2f}','80x80 BGR'][j])
    fig.tight_layout();fig.savefig(figure_dir/'celeba_crop_gallery.png',dpi=130);plt.close(fig)
def gallery_lcc():
    fig,axes=plt.subplots(2,3,figsize=(10,6));rng=np.random.default_rng(101)
    for i in range(2):
        row=valid[valid.label.astype(int)==i].iloc[int(rng.integers((valid.label.astype(int)==i).sum()))]
        image=cv2.imread(str(lcc_root/str(row.path)));rec=lcc_records[str(row.path)]
        patch,box,_=mini_crop_bgr(image,rec['bbox_xyxy'])
        overlay=cv2.cvtColor(image,cv2.COLOR_BGR2RGB);cv2.rectangle(overlay,tuple(map(int,rec['bbox_xyxy'][:2])),tuple(map(int,rec['bbox_xyxy'][2:])),(255,0,0),2)
        for j,v in enumerate([cv2.cvtColor(image,cv2.COLOR_BGR2RGB),overlay,cv2.cvtColor(patch,cv2.COLOR_BGR2RGB)]):
            axes[i,j].imshow(v);axes[i,j].axis('off');axes[i,j].set_title(f'LCC {i}: '+['raw','bbox','2.7 to 80'][j])
    fig.tight_layout();fig.savefig(figure_dir/'lcc_crop_gallery.png',dpi=130);plt.close(fig)
gallery_celeba();gallery_lcc()
if any(not (celeba_root/k).is_file() for k in selected):raise FileNotFoundError('Selected CelebA image missing')
config={'schema':'minifasnet_final_resource_config_v1','data_seed':DATA_SEED,'test_seed':TEST_SEED,
    'sizes':{'train':TRAIN_N,'val':VAL_N,'test':TEST_N,'scale_train':SCALE_TRAIN_N,'scale_val':SCALE_VAL_N},
    'subject_policy':'E1 v5.3 official-Test overlap removal and 64-candidate GroupShuffleSplit; exact stratified subsamples',
    'input':{'scale':2.7,'size':[80,80],'channels':'BGR','range':'[0,1]'},'celeba_bbox_policy':policy,'lcc_policy':lcc_policy}
strict_json_write(root/'config.json',config)
provenance={'source_hashes':source_hashes,'manifest_fingerprints':fingerprints,'celeba_bbox_sha256':sha256_file(cache_path),
    'lcc_manifest_sha256':sha256_file(lcc_manifest),'lcc_bbox_sha256':sha256_file(lcc_cache_path),
    'train_filter_audit_sha256':sha256_file(audit_path),'lcc_audit':lcc_audit}
strict_json_write(root/'provenance.json',provenance)
shutil.copy2(manifest_dir/'manifest_summary.json',root/'manifest_summary.json')
shutil.copy2(manifest_dir/'manifest_fingerprints.json',root/'manifest_fingerprints.json')
report=['# Frozen MiniFASNet final resources','',f'Official Train/Test populations: {len(train_meta)} / {len(test_meta)}.',
    f'Metadata SHA: {source_hashes["train_label_json"]} / {source_hashes["test_label_json"]}.',
    'Sampling: E1 subject overlap removal, subject-disjoint GroupShuffleSplit, stratified exact 100K Train/15K Val; official Test stratified 5K seed 4243. Scale subsets are stratified nested samples.',
    f'Counts: { {k:v["sample_count"] for k,v in fingerprints.items()} }',
    f'Class counts: { {k:v["class_counts"] for k,v in fingerprints.items()} }',
    f'SCRFD policy: {policy}; status counts: {dict(statuses)}.',
    f'BBox SHA: {sha256_file(cache_path)}. LCC: {lcc_audit}.',
    f'Train-only hygiene audit: {train_filter_audit}; audit SHA {sha256_file(audit_path)}.',
    'Crop policy: localized face bbox then 2.7x MiniFASNet crop, direct 80x80 BGR float [0,1].',
    'Limitations: subject split approximates E1 group fraction before exact stratified sampling; fallback annotation is retained; LCC only measures external stress.']
(root/'RESOURCE_REPORT.md').write_text('\n\n'.join(report)+'\n')


3550

In [9]:
ZIP_PATH=Path('/kaggle/working/minifasnet_final_resources_v1.zip')
write_zip(root,ZIP_PATH)
print('ZIP:',ZIP_PATH,'MB:',ZIP_PATH.stat().st_size/1e6)
display(FileLink(str(ZIP_PATH)))


ZIP: /kaggle/working/minifasnet_final_resources_v1.zip MB: 15.713172


/kaggle/working/minifasnet_final_resources_v1.zip